# 26 - Sesion especial: SQL con Python (comandos basicos, JOIN, GROUP BY y merge)

En esta sesion vamos a construir una base de datos artificial y usarla para practicar SQL desde Python.
La idea es que veas teoria corta y luego codigo ejecutable en cada bloque.

## Ruta de la sesion

1. Objetivo y mapa mental SQL + Python.
2. Crear una base artificial en SQLite.
3. Comandos basicos (`SELECT`, `WHERE`, `ORDER BY`, `LIMIT`, `INSERT`, `UPDATE`, `DELETE`).
4. `JOIN` para cruzar tablas.
5. `GROUP BY` y agregaciones.
6. `merge` y `groupby` en pandas (equivalencia mental con SQL).
7. Mini reto resuelto.
8. Ejercicios de pensamiento.

## 1) Objetivo y mapa mental

**SQL** te sirve para consultar y transformar datos tabulares en una base de datos.
**Python** te sirve para automatizar, integrar y analizar resultados.
**pandas** te da una capa de analisis en memoria con sintaxis muy productiva.

Idea clave:
- Piensa en SQL como lenguaje declarativo: dices *que* quieres.
- Piensa en pandas como operaciones encadenadas sobre DataFrames.
- `JOIN` en SQL y `merge` en pandas son primos cercanos.

In [ ]:
import sqlite3
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

## 2) Crear base de datos artificial (SQLite en memoria)

Usaremos `sqlite3` (incluido en Python) para no depender de instalaciones extra.

In [ ]:
conn = sqlite3.connect(":memory:")

def q(sql: str, params=None) -> pd.DataFrame:
    """Ejecuta una consulta SQL y regresa DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)

In [ ]:
clientes_df = pd.DataFrame(
    [
        (1, "Ana", "CDMX", "PyME"),
        (2, "Luis", "Guadalajara", "Enterprise"),
        (3, "Marta", "Monterrey", "PyME"),
        (4, "Jose", "CDMX", "Startup"),
        (5, "Sofia", "Puebla", "Enterprise"),
        (6, "Diego", "Queretaro", "Startup"),
        (7, "Elena", "Merida", "PyME"),
    ],
    columns=["id_cliente", "nombre", "ciudad", "segmento"],
)

productos_df = pd.DataFrame(
    [
        (101, "Laptop", "tecnologia", 24000),
        (102, "Mouse", "tecnologia", 450),
        (103, "Silla", "oficina", 3200),
        (104, "Escritorio", "oficina", 5800),
        (105, "Monitor", "tecnologia", 4100),
        (106, "Lampara", "oficina", 900),
    ],
    columns=["id_producto", "nombre_producto", "categoria", "precio"],
)

ventas_df = pd.DataFrame(
    [
        (1001, "2026-01-05", 1, 101, 1, "web"),
        (1002, "2026-01-06", 1, 102, 2, "tienda"),
        (1003, "2026-01-07", 2, 105, 3, "web"),
        (1004, "2026-01-08", 3, 103, 2, "distribuidor"),
        (1005, "2026-01-08", 4, 104, 1, "web"),
        (1006, "2026-01-09", 4, 102, 5, "web"),
        (1007, "2026-01-10", 5, 101, 1, "tienda"),
        (1008, "2026-01-11", 6, 106, 4, "distribuidor"),
        (1009, "2026-01-11", 2, 103, 1, "web"),
        (1010, "2026-01-12", 3, 105, 2, "tienda"),
        (1011, "2026-01-13", 5, 102, 3, "web"),
        (1012, "2026-01-14", 1, 106, 1, "web"),
    ],
    columns=["id_venta", "fecha", "id_cliente", "id_producto", "cantidad", "canal"],
)

clientes_df.to_sql("clientes", conn, if_exists="replace", index=False)
productos_df.to_sql("productos", conn, if_exists="replace", index=False)
ventas_df.to_sql("ventas", conn, if_exists="replace", index=False)

In [ ]:
q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")

In [ ]:
q("SELECT * FROM clientes ORDER BY id_cliente")

## 3) Comandos basicos de SQL

### 3.1 `SELECT` + columnas
`SELECT` trae columnas de una tabla.

In [ ]:
q(
    """
    SELECT id_cliente, nombre, ciudad
    FROM clientes
    ORDER BY id_cliente
    """
)

### 3.2 `WHERE` para filtrar

Usa condiciones logicas (`=`, `>`, `<`, `IN`, `LIKE`, `BETWEEN`).

In [ ]:
q(
    """
    SELECT id_producto, nombre_producto, categoria, precio
    FROM productos
    WHERE categoria = 'tecnologia' AND precio >= 1000
    ORDER BY precio DESC
    """
)

In [ ]:
q(
    """
    SELECT id_cliente, nombre, ciudad
    FROM clientes
    WHERE ciudad IN ('CDMX', 'Puebla')
    ORDER BY nombre
    """
)

In [ ]:
q(
    """
    SELECT id_producto, nombre_producto, precio
    FROM productos
    WHERE nombre_producto LIKE 'M%'
    """
)

### 3.3 `ORDER BY` + `LIMIT`

Ordena resultados y toma solo las primeras filas.

In [ ]:
q(
    """
    SELECT id_venta, fecha, id_cliente, cantidad
    FROM ventas
    ORDER BY cantidad DESC, fecha ASC
    LIMIT 5
    """
)

### 3.4 `DISTINCT`, alias (`AS`) y columnas calculadas

In [ ]:
q(
    """
    SELECT DISTINCT ciudad
    FROM clientes
    ORDER BY ciudad
    """
)

In [ ]:
q(
    """
    SELECT
        id_venta,
        cantidad,
        id_producto AS sku,
        cantidad * 10 AS puntos_demo
    FROM ventas
    ORDER BY id_venta
    LIMIT 5
    """
)

### 3.5 `INSERT`, `UPDATE`, `DELETE`

Estos comandos modifican datos. En proyectos reales, siempre valida con `WHERE`.

In [ ]:
# INSERT
conn.execute(
    """
    INSERT INTO clientes (id_cliente, nombre, ciudad, segmento)
    VALUES (8, 'Rocio', 'Toluca', 'PyME')
    """
)

# UPDATE
conn.execute(
    """
    UPDATE productos
    SET precio = ROUND(precio * 1.05, 0)
    WHERE categoria = 'oficina'
    """
)

# DELETE (borramos solo registro temporal)
conn.execute(
    """
    INSERT INTO productos (id_producto, nombre_producto, categoria, precio)
    VALUES (999, 'Temporal', 'demo', 1)
    """
)
conn.execute("DELETE FROM productos WHERE id_producto = 999")

conn.commit()

In [ ]:
q("SELECT * FROM clientes WHERE id_cliente = 8")

In [ ]:
q("SELECT * FROM productos ORDER BY id_producto")

## 4) JOIN: unir tablas

`JOIN` permite combinar informacion relacionada en multiples tablas.

Regla mental:
- `INNER JOIN`: solo coincidencias en ambas tablas.
- `LEFT JOIN`: todo lo de la tabla izquierda, aunque no haya coincidencia.

In [ ]:
sql_inner = """
SELECT
    v.id_venta,
    v.fecha,
    c.nombre AS cliente,
    c.ciudad,
    p.nombre_producto AS producto,
    p.categoria,
    v.cantidad,
    p.precio,
    v.cantidad * p.precio AS importe
FROM ventas v
INNER JOIN clientes c ON v.id_cliente = c.id_cliente
INNER JOIN productos p ON v.id_producto = p.id_producto
ORDER BY v.id_venta
"""

q(sql_inner)

In [ ]:
sql_left = """
SELECT
    c.id_cliente,
    c.nombre,
    c.ciudad,
    COUNT(v.id_venta) AS numero_compras
FROM clientes c
LEFT JOIN ventas v ON c.id_cliente = v.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY numero_compras DESC, c.id_cliente
"""

q(sql_left)

In [ ]:
# Anti-join: clientes sin compras
q(
    """
    SELECT c.id_cliente, c.nombre, c.ciudad
    FROM clientes c
    LEFT JOIN ventas v ON c.id_cliente = v.id_cliente
    WHERE v.id_venta IS NULL
    ORDER BY c.id_cliente
    """
)

## 5) `GROUP BY` + agregaciones

`GROUP BY` agrupa filas para calcular metricas por categoria, ciudad, canal, etc.
Funciones tipicas: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`.

In [ ]:
sql_group_ciudad = """
SELECT
    c.ciudad,
    SUM(v.cantidad * p.precio) AS ingresos_totales,
    COUNT(DISTINCT v.id_cliente) AS clientes_unicos,
    COUNT(*) AS lineas_venta
FROM ventas v
JOIN clientes c ON v.id_cliente = c.id_cliente
JOIN productos p ON v.id_producto = p.id_producto
GROUP BY c.ciudad
ORDER BY ingresos_totales DESC
"""

q(sql_group_ciudad)

In [ ]:
sql_group_having = """
SELECT
    p.categoria,
    v.canal,
    ROUND(AVG(v.cantidad * p.precio), 2) AS ticket_promedio,
    SUM(v.cantidad) AS unidades
FROM ventas v
JOIN productos p ON v.id_producto = p.id_producto
GROUP BY p.categoria, v.canal
HAVING SUM(v.cantidad) >= 3
ORDER BY ticket_promedio DESC
"""

q(sql_group_having)

## 6) `groupby` en pandas (equivalente mental)

En pandas, la logica es similar: agrupar y agregar.

In [ ]:
clientes = q("SELECT * FROM clientes")
productos = q("SELECT * FROM productos")
ventas = q("SELECT * FROM ventas")

ventas_detalle = (
    ventas
    .merge(clientes[["id_cliente", "nombre", "ciudad", "segmento"]], on="id_cliente", how="left")
    .merge(productos[["id_producto", "nombre_producto", "categoria", "precio"]], on="id_producto", how="left")
)

ventas_detalle["importe"] = ventas_detalle["cantidad"] * ventas_detalle["precio"]
ventas_detalle.head()

In [ ]:
(
    ventas_detalle
    .groupby("ciudad", as_index=False)
    .agg(
        ingresos_totales=("importe", "sum"),
        clientes_unicos=("id_cliente", "nunique"),
        lineas_venta=("id_venta", "count"),
    )
    .sort_values("ingresos_totales", ascending=False)
)

## 7) `merge` en pandas (equivalente de `JOIN`)

`pd.merge()` replica la idea de `JOIN` y te deja elegir el tipo (`inner`, `left`, etc.).

In [ ]:
# INNER merge
pd.merge(
    ventas,
    clientes[["id_cliente", "nombre", "ciudad"]],
    on="id_cliente",
    how="inner",
).head()

In [ ]:
# LEFT merge + indicador de procedencia
left_demo = pd.merge(
    clientes[["id_cliente", "nombre"]],
    ventas[["id_cliente", "id_venta"]],
    on="id_cliente",
    how="left",
    indicator=True,
)

left_demo.head(12)

In [ ]:
left_demo["_merge"].value_counts()

## 8) Mini reto resuelto (SQL y pandas)

Pregunta: **que clientes tienen al menos 2 compras y cual es su ticket promedio?**

In [ ]:
q(
    """
    SELECT
        c.id_cliente,
        c.nombre,
        COUNT(v.id_venta) AS compras,
        ROUND(AVG(v.cantidad * p.precio), 2) AS ticket_promedio
    FROM ventas v
    JOIN clientes c ON v.id_cliente = c.id_cliente
    JOIN productos p ON v.id_producto = p.id_producto
    GROUP BY c.id_cliente, c.nombre
    HAVING COUNT(v.id_venta) >= 2
    ORDER BY ticket_promedio DESC
    """
)

In [ ]:
ticket_cliente = (
    ventas_detalle
    .groupby(["id_cliente", "nombre"], as_index=False)
    .agg(
        compras=("id_venta", "count"),
        ticket_promedio=("importe", "mean"),
    )
)

ticket_cliente = ticket_cliente[ticket_cliente["compras"] >= 2].copy()
ticket_cliente["ticket_promedio"] = ticket_cliente["ticket_promedio"].round(2)
ticket_cliente.sort_values("ticket_promedio", ascending=False)

## 9) Ejercicios de pensamiento (sin solucion inmediata)

Objetivo: no solo correr consultas, sino justificar decisiones tecnicas.

### Ejercicio 1: Top productos por ciudad

Construye una consulta SQL que regrese el **top 2 de productos por ingresos** en cada ciudad.

Pistas:
- Necesitas `JOIN` de las 3 tablas.
- Agregar por `ciudad` y `producto`.
- Ordenar por ingresos dentro de cada ciudad (puedes intentar con `ROW_NUMBER()` si quieres subir nivel).

In [ ]:
# Escribe aqui tu solucion SQL

### Ejercicio 2: Clientes con concentracion de compra

Detecta clientes que compran solo una categoria (`tecnologia` u `oficina`).

Preguntas para pensar:
- Que estrategia comercial aplicarias a cada grupo?
- Como mediras si la estrategia funciono?

In [ ]:
# Escribe aqui tu solucion SQL o pandas

### Ejercicio 3: `INNER JOIN` vs `LEFT JOIN`

1. Corre una version con `INNER JOIN` y otra con `LEFT JOIN` entre `clientes` y `ventas`.
2. Explica con tus palabras por que cambian los resultados.
3. Define un caso de negocio donde usar `LEFT JOIN` sea obligatorio.

In [ ]:
# Escribe aqui ambas consultas y tu explicacion

### Ejercicio 4: Traduccion SQL <-> pandas

Toma esta pregunta: *ingreso total por segmento y canal*.

Tu tarea:
- Resolverla en SQL.
- Resolverla en pandas (`merge` + `groupby`).
- Comparar legibilidad, control y facilidad de mantenimiento.

In [ ]:
# SQL

# pandas

### Ejercicio 5: Calidad de datos

Imagina que llegan ventas con:
- `id_cliente` inexistente,
- `cantidad` negativa,
- `fecha` nula.

Disena una validacion minima antes de cargar datos a produccion.
Explica que regla aplicas, por que, y que harias con registros invalidos.

In [ ]:
# Escribe aqui tus reglas de validacion (SQL o Python)

## 10) Cierre

Si entiendes bien esta sesion, ya puedes:
- modelar consultas SQL utiles para analisis,
- cruzar tablas con `JOIN`,
- resumir informacion con `GROUP BY`,
- llevar la misma logica a pandas con `merge` y `groupby`.

Siguiente paso recomendado: practicar con una base un poco mas grande (por ejemplo, 100k filas simuladas) y medir rendimiento.